## Global Power Plant Database Analysis

This notebook will guide you through the analysis of the 'Global Power Plant Database' using NumPy, Pandas, and Matplotlib, as requested.

### 1. Data Import and Cleaning

In [36]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats # Added for statistical tests

# Set plotting style for better aesthetics
sns.set_style('whitegrid')

#### Download the Dataset

First, we'll download the dataset directly from the provided URL.

In [37]:
# Define the URL for the dataset
dataset_url = 'https://raw.githubusercontent.com/wri/global-power-plant-database/main/data/gppd_120_jan2024.csv'

# Download the dataset using pandas read_csv directly from the URL
try:
    df = pd.read_csv(dataset_url)
    print("Dataset loaded successfully!")
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Please ensure the URL is correct and you have an active internet connection.")

Error loading dataset: HTTP Error 404: Not Found
Please ensure the URL is correct and you have an active internet connection.


#### Initial Data Inspection

Let's inspect the first few rows, check the data types, and get a summary of missing values to understand the dataset's structure and quality.

In [15]:
# Display the first 5 rows of the DataFrame
display(df.head())

# Display general information about the DataFrame, including data types and non-null values
print("\n--- DataFrame Info ---")
df.info()

# Display the number of missing values per column
print("\n--- Missing Values Count ---")
display(df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False))

""



--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame

--- Missing Values Count ---


,0


In [24]:
# 2. ANALYSE EXPLORATOIRE DES DONNÉES (EDA)
# ==========================================
print("""
--- 2. Analyse Exploratoire ---""")

if df.empty:
    print("Error: DataFrame is empty. Please ensure the dataset was loaded successfully in the previous steps.")
else:
    # Statistiques clés avec Pandas
    print("""Statistiques de la capacité (MW) :
    """, df['capacity_mw'].describe())

    # Répartition par pays (Top 5) et type de carburant
    print("""
    Top 5 des pays producteurs :
    """, df['country_long'].value_counts().head())
    print("""
    Répartition par type de carburant :
    """, df['primary_fuel'].value_counts())


--- 2. Analyse Exploratoire ---
Error: DataFrame is empty. Please ensure the dataset was loaded successfully in the previous steps.


In [28]:
# ==========================================
# 3. ANALYSE STATISTIQUE & TEST D'HYPOTHÈSE
# ==========================================
print("\n--- 3. Analyse Statistique ---")

if df.empty:
    print("Error: DataFrame is empty. Please ensure the dataset was loaded successfully in the previous steps (cell 35932a64).")
else:
    # Extraction des capacités en tableaux NumPy pour deux carburants majeurs (ex: Hydro vs Solar)
    # We need to ensure 'Hydro' and 'Solar' fuels exist and handle potential NaNs
    hydro_cap = df[df['primary_fuel'] == 'Hydro']['capacity_mw'].dropna().to_numpy()
    solar_cap = df[df['primary_fuel'] == 'Solar']['capacity_mw'].dropna().to_numpy()

    if len(hydro_cap) == 0 or len(solar_cap) == 0:
        print("Not enough data for Hydro or Solar power plants to perform statistical analysis.")
    else:
        # Statistiques avancées avec NumPy
        print(f"Hydro - Moyenne: {np.mean(hydro_cap):.2f} MW, Médiane: {np.median(hydro_cap):.2f} MW, Écart-type: {np.std(hydro_cap):.2f}")
        print(f"Solar - Moyenne: {np.mean(solar_cap):.2f} MW, Médiane: {np.median(solar_cap):.2f} MW, Écart-type: {np.std(solar_cap):.2f}")

        # Test d'hypothèse (T-test de Student pour échantillons indépendants)
        # Using equal_var=False for Welch's t-test, which does not assume equal population variance
        t_stat, p_val = stats.ttest_ind(hydro_cap, solar_cap, equal_var=False)
        print(f"\nTest T entre Hydro et Solar : t = {t_stat:.3f}, p-value = {p_val:.3e}")
        if p_val < 0.05:
            print("👉 Rejet de l'hypothèse nulle : La différence de puissance moyenne est statistiquement significative.")
        else:
            print("👉 Échec du rejet de l'hypothèse nulle : Pas de différence significative.")


--- 3. Analyse Statistique ---
Error: DataFrame is empty. Please ensure the dataset was loaded successfully in the previous steps (cell 35932a64).


In [31]:
# ==========================================
# 4. ANALYSE DES SÉRIES CHRONOLOGIQUES
# ==========================================
print("\n--- 4. Analyse Chronologique ---")

if df.empty:
    print("Error: DataFrame is empty. Please ensure the dataset was loaded successfully in previous steps (cell 35932a64).")
else:
    # Filtrer les années valides (ex: après 1950) via un masque NumPy
    # Ensure 'commissioning_year' is numeric and handle potential NaNs
    df['commissioning_year'] = pd.to_numeric(df['commissioning_year'], errors='coerce')
    valid_years_mask = (df['commissioning_year'].to_numpy() >= 1950) & (df['commissioning_year'].to_numpy() <= 2021)
    df_chrono = df[valid_years_mask]

    if df_chrono.empty:
        print("No data available for the specified commissioning year range (1950-2021).")
    else:
        # Évolution de la diversité : Nombre de centrales construites par an et par carburant
        evolution = df_chrono.groupby(['commissioning_year', 'primary_fuel']).size().unstack(fill_value=0)
        print("Aperçu de l'évolution temporelle :\n", evolution.tail())


--- 4. Analyse Chronologique ---
Error: DataFrame is empty. Please ensure the dataset was loaded successfully in previous steps (cell 35932a64).


In [38]:
# ==========================================
# 5. VISUALISATION AVANCÉE
# ==========================================
print("\n--- 5. Génération des Graphiques ---")

if df.empty:
    print("Error: DataFrame is empty. Please ensure the dataset was loaded successfully in cell 35932a64 and subsequent analysis cells ran correctly.")
elif 'evolution' not in locals() and 'evolution' not in globals():
    print("Error: The 'evolution' DataFrame was not created. Please ensure cell So5CzP0yOcDg (Time Series Analysis) ran successfully.")
else:
    # Graphique 1 : Évolution temporelle des carburants
    plt.figure()
    evolution[['Hydro', 'Solar', 'Wind', 'Coal', 'Gas']].plot(kind='area', stacked=True, alpha=0.7)
    plt.title("Évolution des constructions de centrales par type de carburant (1950-2021)")
    plt.xlabel("Année de mise en service")
    plt.ylabel("Nombre de centrales")
    plt.legend(title="Carburant")
    plt.tight_layout()
    plt.show()

    # Graphique 2 : Cartographie globale de la capacité
    plt.figure(figsize=(14, 7))
    scatter = plt.scatter(df['longitude'], df['latitude'], c=df['capacity_mw'],
                          cmap='viridis', norm=plt.cm.colors.LogNorm(), alpha=0.6, s=5)
    plt.colorbar(scatter, label='Capacité de la centrale (MW) - Échelle Log')
    plt.title("Répartition géographique mondiale et capacité des centrales électriques")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.tight_layout()
    plt.show()


--- 5. Génération des Graphiques ---
Error: DataFrame is empty. Please ensure the dataset was loaded successfully in cell 35932a64 and subsequent analysis cells ran correctly.


In [42]:
# ==========================================
# 6. OPÉRATIONS MATRICIELLES & INTÉGRATION
# ==========================================
print("\n--- 6. Opérations Matricielles & Intégration ---")

if df.empty:
    print("Error: DataFrame is empty. Please ensure the dataset was loaded successfully in cell 35932a64.")
else:
    # Sélection de variables numériques pour une analyse en composantes principales (ACP / Corrélation)
    # We need to ensure these columns exist and handle NaNs
    required_cols = ['capacity_mw', 'latitude', 'longitude']
    if not all(col in df.columns for col in required_cols):
        print(f"Error: Missing one or more required columns ({required_cols}) in the DataFrame.")
    else:
        features = df[required_cols].dropna().to_numpy()

        if features.shape[0] == 0:
            print("No valid data points after dropping NaNs for feature columns.")
        else:
            # Centrer et réduire la matrice de données
            features_centered = features - np.mean(features, axis=0)
            covariance_matrix = np.cov(features_centered, rowvar=False)

            # Calcul des valeurs propres et vecteurs propres
            valeurs_propres, vecteurs_propres = np.linalg.eig(covariance_matrix)
            print("Matrice de covariance :\n", covariance_matrix)
            print("\nValeurs propres (variance expliquée) :\n", valeurs_propres)
            print("\nVecteurs propres (directions principales) :\n", vecteurs_propres)


--- 6. Opérations Matricielles & Intégration ---
Error: DataFrame is empty. Please ensure the dataset was loaded successfully in cell 35932a64.
